# HUC-8 / 10 / 12 feasibility v2 — contributing network only

Rebuilds the scale-justification screen using the **exact measurement network the load analysis used**,
per review Point 3:

- **flow:** only the gauges that actually returned usable daily discharge (the ~166 contributing
  gauges from the pipeline cache), not all ~261 queried gauge locations;
- **nitrate:** the pipeline-filtered DNR + USGS sites, deduplicated to site locations.

Two bounds are reported per scale:
1. **Spatial** — a unit contains a contributing gauge *and* a nitrate site (upper bound).
2. **Same-month** — a unit has at least one month with *both* a flow observation and a nitrate sample
   (the actual load-estimable condition; a tighter, lower bound).

Reads `cache/` and `data/` from the pipeline run.

## 0. Setup

In [ ]:
# ==============================================================
# HUC Scale Justification v5 — FINAL FROZEN VERSION
# HUC-8 vs HUC-10 vs HUC-12
#
# Uses:
#   1. Exact Pipeline v4 nitrate filtering
#   2. Contributing USGS gauges only
#   3. >=25 valid discharge days per gauge-month
#   4. Same-HUC, same-year, same-month flow–nitrate pairing
#   5. Complete HUC-year = exactly 12 paired months
#   6. Study-period annual feasibility = 2001–2019
# ==============================================================

import os
import warnings
from pathlib import Path

import numpy as np
import pandas as pd
import geopandas as gpd

warnings.filterwarnings("ignore")

pd.set_option("display.max_columns", 50)
pd.set_option("display.width", 180)







## 1. Watershed layers (HUC-8/10/12), clipped to Iowa

In [10]:
# ==============================================================
# 0. CONFIGURATION
# ==============================================================

CRS = "EPSG:26915"

DATA = Path("data")
CACHE = Path("cache")
OUT = Path("out")

OUT.mkdir(parents=True, exist_ok=True)

START_YEAR = 2000
END_YEAR = 2020

ANALYSIS_START = 2001
ANALYSIS_END = 2019

MIN_DAYS_PER_SITE_MONTH = 25
MIN_HUC_AREA_HA = 50
MAX_NITRATE_MGL = 100.0

EXPECTED_NITRATE_SAMPLES = 38_335
EXPECTED_DNR_SAMPLES = 30_676
EXPECTED_USGS_SAMPLES = 7_659


def norm_huc(series, digits):
    """Normalize HUC identifiers while preserving leading zeros."""
    return (
        series.astype(str)
        .str.strip()
        .str.replace(r"\.0$", "", regex=True)
        .str.extract(r"(\d+)", expand=False)
        .str.zfill(digits)
    )


def clean_county_name(series):
    """Standardize Iowa county names."""
    fixes = {
        "Obrien": "O'Brien",
        "O Brien": "O'Brien",
        "Obrien County": "O'Brien",
        "O'brien": "O'Brien",
    }

    return (
        series.astype(str)
        .str.strip()
        .str.replace(r"\s+County$", "", regex=True, case=False)
        .str.title()
        .replace(fixes)
    )


print("=" * 76)
print("HUC-8 / HUC-10 / HUC-12 FINAL MONITORING-FEASIBILITY ANALYSIS")
print("=" * 76)
print(f"Data period: {START_YEAR}-{END_YEAR}")
print(f"Annual feasibility period: {ANALYSIS_START}-{ANALYSIS_END}")
print(f"Minimum valid flow days per gauge-month: {MIN_DAYS_PER_SITE_MONTH}")

HUC-8 / HUC-10 / HUC-12 FINAL MONITORING-FEASIBILITY ANALYSIS
Data period: 2000-2020
Annual feasibility period: 2001-2019
Minimum valid flow days per gauge-month: 25


## 2. Contributing flow gauges (166), not all queried

The pipeline cached the retrieved daily discharge. Unique `site_no` in that file are the gauges that
actually contributed; coordinates come from the gauge metadata.

In [11]:
# ==============================================================
# 1. IOWA COUNTY BOUNDARY
# ==============================================================

county_candidates = [
    DATA / "Iowa_County_Boundaries" / "IowaCounties.shp",
    DATA / "IowaCounties.shp",
    DATA / "iowa_counties.geojson",
]

county_path = next((p for p in county_candidates if p.exists()), None)

if county_path is None:
    raise FileNotFoundError(
        "Iowa county boundary file was not found. Expected one of:\n"
        + "\n".join(str(p) for p in county_candidates)
    )

counties_raw = gpd.read_file(county_path)

county_col = next(
    (
        c for c in [
            "CountyName",
            "unitName",
            "COUNTY",
            "COUNTYNAME",
            "NAME",
            "Name",
            "County",
        ]
        if c in counties_raw.columns
    ),
    None,
)

if county_col is None:
    raise KeyError(
        "County-name column could not be identified. "
        f"Available columns: {counties_raw.columns.tolist()}"
    )

counties_raw["CountyName"] = clean_county_name(
    counties_raw[county_col]
)

counties = (
    counties_raw.loc[
        counties_raw.geometry.notna()
        & ~counties_raw.geometry.is_empty,
        ["CountyName", "geometry"],
    ]
    .to_crs(CRS)
    .dissolve(by="CountyName", as_index=False)
)

if counties["CountyName"].nunique() != 99:
    raise ValueError(
        f"Expected 99 Iowa counties, found "
        f"{counties['CountyName'].nunique()}."
    )

iowa = counties[["geometry"]].dissolve().reset_index(drop=True)
iowa_area_ha = float(iowa.geometry.area.iloc[0] / 10_000)

print(f"\nIowa counties: {counties.CountyName.nunique()}")
print(f"Iowa analysis area: {iowa_area_ha / 258.999:.1f} mi²")


Iowa counties: 99
Iowa analysis area: 56254.4 mi²


In [13]:
# ==============================================================
# 2. LOAD AND CLIP HUC-8, HUC-10, AND HUC-12
# ==============================================================

scale_specs = {
    "HUC-8": {
        "digits": 8,
        "paths": [
            DATA / "WBD_IA" / "WBD_HU_08_IA.shp",
            DATA / "WBD_HU_08_IA.shp",
        ],
        "columns": ["HUC_8", "HUC8", "huc8", "huc_8"],
    },
    "HUC-10": {
        "digits": 10,
        "paths": [
            DATA / "WBD_IA" / "WBD_HU_10_IA.shp",
            DATA / "WBD_HU_10_IA.shp",
        ],
        "columns": ["HUC_10", "HUC10", "huc10", "huc_10"],
    },
    "HUC-12": {
        "digits": 12,
        "paths": [
            DATA / "WBD_IA" / "WBD_HU_12_IA.shp",
            DATA / "WBD_HU_12_IA.shp",
        ],
        "columns": ["HUC_12", "HUC12", "huc12", "huc_12"],
    },
}

SCALES = {}

for scale, specification in scale_specs.items():
    path = next(
        (p for p in specification["paths"] if p.exists()),
        None,
    )

    if path is None:
        raise FileNotFoundError(
            f"{scale} boundary file was not found. Expected one of:\n"
            + "\n".join(str(p) for p in specification["paths"])
        )

    layer = gpd.read_file(path)

    huc_col = next(
        (c for c in specification["columns"] if c in layer.columns),
        None,
    )

    if huc_col is None:
        raise KeyError(
            f"{scale} identifier was not found. "
            f"Available columns: {layer.columns.tolist()}"
        )

    layer["UID"] = norm_huc(
        layer[huc_col],
        specification["digits"],
    )

    layer = (
        layer.loc[
            layer["UID"].notna()
            & layer.geometry.notna()
            & ~layer.geometry.is_empty,
            ["UID", "geometry"],
        ]
        .to_crs(CRS)
        .dissolve(by="UID", as_index=False)
    )

    # Clip every scale to exactly the same Iowa boundary.
    layer = gpd.overlay(
        layer,
        iowa,
        how="intersection",
        keep_geom_type=False,
    )

    layer = layer.loc[
        layer.geometry.notna()
        & ~layer.geometry.is_empty
    ].copy()

    layer["area_ha"] = layer.geometry.area / 10_000

    # Remove small boundary slivers consistently.
    layer = layer.loc[
        layer["area_ha"] > MIN_HUC_AREA_HA
    ].copy()

    layer["area_sqmi"] = layer["area_ha"] / 258.999

    SCALES[scale] = layer

    print(
        f"{scale}: {layer.UID.nunique():,} Iowa-intersecting units; "
        f"median Iowa portion = {layer.area_sqmi.median():.1f} mi²"
    )

HUC-8: 56 Iowa-intersecting units; median Iowa portion = 889.8 mi²
HUC-10: 388 Iowa-intersecting units; median Iowa portion = 127.8 mi²
HUC-12: 1,708 Iowa-intersecting units; median Iowa portion = 32.4 mi²


In [14]:
# ==============================================================
# 3. FLOW: EXACT CONTRIBUTING NETWORK AND QUALIFYING SITE-MONTHS
# ==============================================================

daily_path = CACHE / "usgs_daily.parquet"
site_path = CACHE / "usgs_sites.csv"

if not daily_path.exists():
    raise FileNotFoundError(
        f"Missing {daily_path}. Run Iowa_N_Load_Pipeline_v4 first."
    )

if not site_path.exists():
    raise FileNotFoundError(
        f"Missing {site_path}. Run Iowa_N_Load_Pipeline_v4 first."
    )

daily = pd.read_parquet(daily_path)
sites = pd.read_csv(
    site_path,
    dtype={"site_no": str},
    low_memory=False,
)

required_daily = {"site_no", "Date", "flow_cfs"}
missing_daily = required_daily - set(daily.columns)

if missing_daily:
    raise KeyError(
        f"Daily-flow cache is missing: {sorted(missing_daily)}"
    )

required_sites = {
    "site_no",
    "dec_lat_va",
    "dec_long_va",
}

missing_sites = required_sites - set(sites.columns)

if missing_sites:
    raise KeyError(
        f"Gauge metadata is missing: {sorted(missing_sites)}"
    )

daily["site_no"] = daily["site_no"].astype(str).str.strip()
daily["Date"] = pd.to_datetime(
    daily["Date"],
    errors="coerce",
).dt.tz_localize(None)

daily["flow_cfs"] = pd.to_numeric(
    daily["flow_cfs"],
    errors="coerce",
)

daily = daily.loc[
    daily["Date"].notna()
    & daily["Date"].dt.year.between(
        START_YEAR,
        END_YEAR,
    )
    & daily["flow_cfs"].notna()
    & np.isfinite(daily["flow_cfs"])
    & (daily["flow_cfs"] >= 0)
].copy()

daily["Year"] = daily["Date"].dt.year
daily["Month"] = daily["Date"].dt.month

sites["site_no"] = sites["site_no"].astype(str).str.strip()
sites["dec_lat_va"] = pd.to_numeric(
    sites["dec_lat_va"],
    errors="coerce",
)
sites["dec_long_va"] = pd.to_numeric(
    sites["dec_long_va"],
    errors="coerce",
)

if "drain_area_mi2" in sites.columns:
    sites["drain_area_mi2"] = pd.to_numeric(
        sites["drain_area_mi2"],
        errors="coerce",
    )

    sites = sites.loc[
        sites["drain_area_mi2"].notna()
        & (sites["drain_area_mi2"] > 0)
    ].copy()

sites = (
    sites.dropna(
        subset=["dec_lat_va", "dec_long_va"]
    )
    .drop_duplicates(
        subset="site_no",
        keep="first",
    )
)

# Only gauges that actually returned valid daily observations.
contributing_ids = set(daily["site_no"].unique())

gauge_metadata = sites.loc[
    sites["site_no"].isin(contributing_ids)
].copy()

GFLOW_ALL = gpd.GeoDataFrame(
    gauge_metadata,
    geometry=gpd.points_from_xy(
        gauge_metadata["dec_long_va"],
        gauge_metadata["dec_lat_va"],
    ),
    crs="EPSG:4326",
).to_crs(CRS)

# Apply the exact Pipeline v4 >=25-day site-month rule.
flow_site_month = (
    daily.groupby(
        ["site_no", "Year", "Month"],
        as_index=False,
    )
    .agg(
        ndays=("flow_cfs", "size"),
    )
)

flow_site_month = flow_site_month.loc[
    flow_site_month["ndays"]
    >= MIN_DAYS_PER_SITE_MONTH
].copy()

flow_site_month = flow_site_month.merge(
    GFLOW_ALL[["site_no", "geometry"]],
    on="site_no",
    how="inner",
)

FLOW_SITE_MONTH = gpd.GeoDataFrame(
    flow_site_month,
    geometry="geometry",
    crs=CRS,
)

# Spatial gauge network contains only gauges with at least one
# qualifying >=25-day site-month.
FLOW_POINTS = (
    FLOW_SITE_MONTH[
        ["site_no", "geometry"]
    ]
    .drop_duplicates(
        subset="site_no"
    )
    .copy()
)

FLOW_POINTS = gpd.GeoDataFrame(
    FLOW_POINTS,
    geometry="geometry",
    crs=CRS,
)

print("\nFLOW ACCOUNTING")
print(f"Valid daily-flow records: {len(daily):,}")
print(
    f"Gauges with valid daily observations: "
    f"{daily.site_no.nunique():,}"
)
print(
    f"Qualifying gauge-months (>=25 days): "
    f"{len(FLOW_SITE_MONTH):,}"
)
print(
    f"Gauges with at least one qualifying month: "
    f"{FLOW_POINTS.site_no.nunique():,}"
)



FLOW ACCOUNTING
Valid daily-flow records: 1,018,610
Gauges with valid daily observations: 166
Qualifying gauge-months (>=25 days): 33,437
Gauges with at least one qualifying month: 166


In [15]:
# ==============================================================
# 4. NITRATE: EXACT PIPELINE V4 FILTERS
# ==============================================================

# --------------------------------------------------------------
# 4.1 Iowa DNR
# --------------------------------------------------------------

dnr_path = DATA / "concentration_data.csv"

if not dnr_path.exists():
    raise FileNotFoundError(
        f"Missing {dnr_path}."
    )

dnr = pd.read_csv(
    dnr_path,
    low_memory=False,
)

if "type" in dnr.columns:
    dnr = dnr.loc[
        dnr["type"].astype(str).eq("River/Stream")
    ].copy()

if "detect" in dnr.columns:
    dnr = dnr.loc[
        dnr["detect"].eq(True)
    ].copy()

dnr["result"] = pd.to_numeric(
    dnr["result"],
    errors="coerce",
)

dnr["latitude"] = pd.to_numeric(
    dnr["latitude"],
    errors="coerce",
)

dnr["longitude"] = pd.to_numeric(
    dnr["longitude"],
    errors="coerce",
)

dnr = dnr.dropna(
    subset=["result", "latitude", "longitude"]
)

dnr = dnr.loc[
    dnr["result"].between(
        0,
        MAX_NITRATE_MGL,
        inclusive="left",
    )
].copy()

dnr["Date"] = pd.to_datetime(
    dnr["sampleDate"],
    errors="coerce",
)

dnr = dnr.dropna(subset=["Date"])

dnr = dnr.loc[
    dnr["Date"].dt.year.between(
        START_YEAR,
        END_YEAR,
    )
].copy()

dnr_nitrate = dnr[
    ["latitude", "longitude", "Date"]
].copy()

dnr_nitrate["source"] = "DNR"

In [16]:
# --------------------------------------------------------------
# 4.2 USGS discrete nitrate
# --------------------------------------------------------------

usgs_path = CACHE / "usgs_discrete_nitrate.csv"

if not usgs_path.exists():
    raise FileNotFoundError(
        f"Missing {usgs_path}. Run Iowa_N_Load_Pipeline_v4 first."
    )

usgs = pd.read_csv(
    usgs_path,
    low_memory=False,
)

lat_col = next(
    (
        c for c in [
            "LatitudeMeasure",
            "latitude",
            "lat",
        ]
        if c in usgs.columns
    ),
    None,
)

lon_col = next(
    (
        c for c in [
            "LongitudeMeasure",
            "longitude",
            "lon",
        ]
        if c in usgs.columns
    ),
    None,
)

date_col = next(
    (
        c for c in [
            "ActivityStartDate",
            "Date",
            "date",
        ]
        if c in usgs.columns
    ),
    None,
)

if lat_col is None or lon_col is None or date_col is None:
    raise KeyError(
        "USGS nitrate cache does not contain the required "
        "coordinate/date columns."
    )

usgs["result"] = pd.to_numeric(
    usgs.get("ResultMeasureValue"),
    errors="coerce",
)

usgs = usgs.dropna(subset=["result"])

unit_series = (
    usgs.get(
        "ResultMeasure/MeasureUnitCode",
        pd.Series(index=usgs.index, dtype=object),
    )
    .astype(str)
    .str.lower()
    .str.replace(" ", "", regex=False)
)

usgs = usgs.loc[
    unit_series.isin(["mg/l", "mg/lasn"])
    | unit_series.eq("nan")
].copy()

usgs = usgs.loc[
    usgs["result"].between(
        0,
        MAX_NITRATE_MGL,
        inclusive="left",
    )
].copy()

usgs["latitude"] = pd.to_numeric(
    usgs[lat_col],
    errors="coerce",
)

usgs["longitude"] = pd.to_numeric(
    usgs[lon_col],
    errors="coerce",
)

usgs["Date"] = pd.to_datetime(
    usgs[date_col],
    errors="coerce",
)

usgs = usgs.dropna(
    subset=["latitude", "longitude", "Date"]
)

usgs = usgs.loc[
    usgs["Date"].dt.year.between(
        START_YEAR,
        END_YEAR,
    )
].copy()

usgs_nitrate = usgs[
    ["latitude", "longitude", "Date"]
].copy()

usgs_nitrate["source"] = "USGS"

In [18]:
# --------------------------------------------------------------
# 4.3 Pool exact retained observations
# --------------------------------------------------------------

NITRATE = pd.concat(
    [dnr_nitrate, usgs_nitrate],
    ignore_index=True,
)

NITRATE["Year"] = NITRATE["Date"].dt.year
NITRATE["Month"] = NITRATE["Date"].dt.month

GCONC = gpd.GeoDataFrame(
    NITRATE,
    geometry=gpd.points_from_xy(
        NITRATE["longitude"],
        NITRATE["latitude"],
    ),
    crs="EPSG:4326",
).to_crs(CRS)

print("\nNITRATE ACCOUNTING")
print(f"DNR retained samples: {len(dnr_nitrate):,}")
print(f"USGS retained samples: {len(usgs_nitrate):,}")
print(f"Pooled retained samples: {len(GCONC):,}")

assert len(dnr_nitrate) == EXPECTED_DNR_SAMPLES, (
    f"DNR count is {len(dnr_nitrate):,}; expected "
    f"{EXPECTED_DNR_SAMPLES:,} from Pipeline v4."
)

assert len(usgs_nitrate) == EXPECTED_USGS_SAMPLES, (
    f"USGS count is {len(usgs_nitrate):,}; expected "
    f"{EXPECTED_USGS_SAMPLES:,} from Pipeline v4."
)

assert len(GCONC) == EXPECTED_NITRATE_SAMPLES, (
    f"Pooled nitrate count is {len(GCONC):,}; expected "
    f"{EXPECTED_NITRATE_SAMPLES:,} from Pipeline v4."
)


NITRATE ACCOUNTING
DNR retained samples: 30,676
USGS retained samples: 7,659
Pooled retained samples: 38,335


In [19]:
# ==============================================================
# 5. HELPER FUNCTIONS
# ==============================================================

def assign_points_to_units(points, units, extra_columns=None):
    """Assign monitoring points to watershed units."""

    extra_columns = extra_columns or []

    point_columns = [
        c for c in extra_columns
        if c in points.columns
    ] + ["geometry"]

    joined = gpd.sjoin(
        points[point_columns].copy(),
        units[["UID", "geometry"]].copy(),
        how="inner",
        predicate="within",
    )

    return joined.drop(
        columns=["index_right"],
        errors="ignore",
    )


def unit_ids(points, units):
    """Return units containing at least one monitoring location."""

    joined = assign_points_to_units(
        points,
        units,
    )

    return set(
        joined["UID"]
        .dropna()
        .astype(str)
        .unique()
    )


def paired_month_table(units):
    """
    Return unique HUC-year-month records containing both:
      - at least one qualifying >=25-day flow site-month; and
      - at least one valid nitrate observation.
    """

    flow_join = assign_points_to_units(
        FLOW_SITE_MONTH,
        units,
        extra_columns=["Year", "Month"],
    )

    nitrate_join = assign_points_to_units(
        GCONC,
        units,
        extra_columns=["Year", "Month"],
    )

    flow_keys = (
        flow_join.loc[
            flow_join["Year"].between(
                ANALYSIS_START,
                ANALYSIS_END,
            ),
            ["UID", "Year", "Month"],
        ]
        .drop_duplicates()
    )

    nitrate_keys = (
        nitrate_join.loc[
            nitrate_join["Year"].between(
                ANALYSIS_START,
                ANALYSIS_END,
            ),
            ["UID", "Year", "Month"],
        ]
        .drop_duplicates()
    )

    paired = flow_keys.merge(
        nitrate_keys,
        on=["UID", "Year", "Month"],
        how="inner",
        validate="one_to_one",
    )

    return paired


def county_reach(units, supported_uids):
    """Count counties intersecting supported watershed units."""

    supported = units.loc[
        units["UID"].astype(str).isin(
            supported_uids
        ),
        ["UID", "geometry"],
    ].copy()

    if supported.empty:
        return 0

    intersections = gpd.overlay(
        counties[["CountyName", "geometry"]],
        supported,
        how="intersection",
        keep_geom_type=False,
    )

    intersections = intersections.loc[
        intersections.geometry.notna()
        & ~intersections.geometry.is_empty
    ]

    return int(
        intersections["CountyName"].nunique()
    )


def supported_area_percent(units, supported_uids):
    """Percentage of Iowa-clipped scale area in supported units."""

    supported_area = units.loc[
        units["UID"].astype(str).isin(
            supported_uids
        ),
        "area_ha",
    ].sum()

    total_area = units["area_ha"].sum()

    return (
        100 * supported_area / total_area
        if total_area > 0
        else np.nan
    )


In [20]:
# ==============================================================
# 6. SPATIAL MONITORING COVERAGE
# ==============================================================

spatial_rows = []

for scale, units in SCALES.items():
    flow_units = unit_ids(
        FLOW_POINTS,
        units,
    )

    nitrate_units = unit_ids(
        GCONC,
        units,
    )

    both_units = flow_units & nitrate_units

    paired = paired_month_table(units)

    any_paired_units = set(
        paired["UID"]
        .dropna()
        .astype(str)
        .unique()
    )

    spatial_rows.append({
        "scale": scale,
        "units": int(units["UID"].nunique()),
        "median_unit_sqmi": float(
            units["area_sqmi"].median()
        ),
        "with_qualifying_flow": len(flow_units),
        "with_valid_nitrate": len(nitrate_units),
        "with_both_spatial": len(both_units),
        "pct_area_spatial": supported_area_percent(
            units,
            both_units,
        ),
        "with_any_paired_month": len(
            any_paired_units
        ),
        "pct_area_any_paired_month": (
            supported_area_percent(
                units,
                any_paired_units,
            )
        ),
        "counties_any_paired_month": county_reach(
            units,
            any_paired_units,
        ),
        "paired_huc_months_2001_2019": len(paired),
    })

SPATIAL = pd.DataFrame(spatial_rows)

print("\n" + "=" * 76)
print("SPATIAL AND ANY-PAIRED-MONTH COVERAGE")
print("=" * 76)
print(SPATIAL.round(1).to_string(index=False))


SPATIAL AND ANY-PAIRED-MONTH COVERAGE
 scale  units  median_unit_sqmi  with_qualifying_flow  with_valid_nitrate  with_both_spatial  pct_area_spatial  with_any_paired_month  pct_area_any_paired_month  counties_any_paired_month  paired_huc_months_2001_2019
 HUC-8     56             889.8                    45                  52                 44              92.4                     43                       91.3                         98                         8438
HUC-10    388             127.8                   125                 299                120              40.5                    116                       39.3                         92                        14050
HUC-12   1708              32.4                   151                 674                122               8.6                    116                        8.1                         73                        10972


In [21]:
# ==============================================================
# 7. PAIRED-MONTH FEASIBILITY SPECTRUM
# ==============================================================

spectrum_rows = []
annual_rows = []

criteria = [
    (">=1 paired month", 1),
    (">=8 paired months", 8),
    ("12 paired months (complete)", 12),
]

for scale, units in SCALES.items():
    paired = paired_month_table(units)

    paired_counts = (
        paired.groupby(
            ["UID", "Year"],
            as_index=False,
        )
        .agg(
            paired_months=("Month", "nunique"),
        )
    )

    for criterion, minimum_months in criteria:
        eligible_huc_years = paired_counts.loc[
            paired_counts["paired_months"]
            >= minimum_months
        ].copy()

        supported_uids = set(
            eligible_huc_years["UID"]
            .dropna()
            .astype(str)
            .unique()
        )

        spectrum_rows.append({
            "criterion": criterion,
            "minimum_paired_months": minimum_months,
            "scale": scale,
            "eligible_huc_years": len(
                eligible_huc_years
            ),
            "supported_units": len(
                supported_uids
            ),
            "pct_area": supported_area_percent(
                units,
                supported_uids,
            ),
            "counties_reached": county_reach(
                units,
                supported_uids,
            ),
        })

    complete_huc_years = paired_counts.loc[
        paired_counts["paired_months"].eq(12)
    ].copy()

    complete_uids = set(
        complete_huc_years["UID"]
        .dropna()
        .astype(str)
        .unique()
    )

    annual_rows.append({
        "scale": scale,
        "paired_huc_months": len(paired),
        "complete_huc_years": len(
            complete_huc_years
        ),
        "units_with_complete_huc_year": len(
            complete_uids
        ),
        "pct_area_with_complete_huc_year": (
            supported_area_percent(
                units,
                complete_uids,
            )
        ),
        "counties_reached_complete_huc_year": (
            county_reach(
                units,
                complete_uids,
            )
        ),
    })

SPECTRUM = pd.DataFrame(spectrum_rows)
ANNUAL = pd.DataFrame(annual_rows)

print("\n" + "=" * 76)
print("PAIRED-MONTH FEASIBILITY SPECTRUM, 2001-2019")
print("=" * 76)

for criterion in SPECTRUM["criterion"].unique():
    print(f"\n{criterion}")

    view = SPECTRUM.loc[
        SPECTRUM["criterion"].eq(criterion),
        [
            "scale",
            "eligible_huc_years",
            "supported_units",
            "pct_area",
            "counties_reached",
        ],
    ]

    print(
        view.round(1).to_string(index=False)
    )

print("\n" + "=" * 76)
print("COMPLETE 12-PAIRED-MONTH HUC-YEAR RESULTS")
print("=" * 76)
print(ANNUAL.round(1).to_string(index=False))


PAIRED-MONTH FEASIBILITY SPECTRUM, 2001-2019

>=1 paired month
 scale  eligible_huc_years  supported_units  pct_area  counties_reached
 HUC-8                 754               43      91.3                98
HUC-10                1435              116      39.3                92
HUC-12                1130              116       8.1                73

>=8 paired months
 scale  eligible_huc_years  supported_units  pct_area  counties_reached
 HUC-8                 722               41      88.9                98
HUC-10                1184               89      30.8                87
HUC-12                 933               77       5.4                61

12 paired months (complete)
 scale  eligible_huc_years  supported_units  pct_area  counties_reached
 HUC-8                 583               38      86.9                98
HUC-10                 814               70      23.9                85
HUC-12                 599               58       4.2                55

COMPLETE 12-PAIRED-MONT

In [22]:
# ==============================================================
# 9. DYNAMIC PAPER-READY TEXT
# ==============================================================

annual_index = ANNUAL.set_index("scale")
spatial_index = SPATIAL.set_index("scale")

required_scales = {"HUC-8", "HUC-10", "HUC-12"}

if not required_scales.issubset(
    set(annual_index.index)
):
    raise ValueError(
        "All three scales were not successfully analyzed."
    )

h8 = annual_index.loc["HUC-8"]
h10 = annual_index.loc["HUC-10"]
h12 = annual_index.loc["HUC-12"]

s8 = spatial_index.loc["HUC-8"]
s10 = spatial_index.loc["HUC-10"]
s12 = spatial_index.loc["HUC-12"]

methods_text = f"""
METHODS — HUC-SCALE JUSTIFICATION

We evaluated HUC-8, HUC-10, and HUC-12 as candidate observational
scales after clipping all watershed layers to the same Iowa boundary.
Streamflow support was restricted to gauges that contributed valid
daily discharge during 2000–2020, and a gauge-month was retained only
when at least {MIN_DAYS_PER_SITE_MONTH} valid daily observations were
available. Nitrate observations were filtered using the same criteria
as the final county-load pipeline, retaining {len(GCONC):,} Iowa DNR
and USGS observations. A paired watershed-month required qualifying
streamflow and nitrate observations in the same hydrologic unit,
calendar year, and month. A complete watershed-year required all
12 months to be paired. Annual feasibility was evaluated over
{ANALYSIS_START}–{ANALYSIS_END}.
""".strip()

results_text = f"""
RESULTS — HUC-SCALE JUSTIFICATION

Spatial co-location of qualifying streamflow and nitrate monitoring
covered {s8['pct_area_spatial']:.1f}% of Iowa at HUC-8,
{s10['pct_area_spatial']:.1f}% at HUC-10, and
{s12['pct_area_spatial']:.1f}% at HUC-12. Under the stricter annual
criterion requiring at least one complete 12-paired-month
watershed-year, supported watersheds represented
{h8['pct_area_with_complete_huc_year']:.1f}%,
{h10['pct_area_with_complete_huc_year']:.1f}%, and
{h12['pct_area_with_complete_huc_year']:.1f}% of Iowa, respectively.
These supported watersheds reached
{int(h8['counties_reached_complete_huc_year'])},
{int(h10['counties_reached_complete_huc_year'])}, and
{int(h12['counties_reached_complete_huc_year'])} of Iowa's
99 counties. The comparison supports retaining HUC-8 as the primary
statewide observational scale among the three candidates; HUC-10 and
HUC-12 provide finer spatial resolution but substantially lower
measurement-supported annual coverage.
""".strip()

reviewer_text = f"""
RESPONSE TO REVIEWER

We agree that HUC-12 provides greater spatial precision in principle.
We therefore compared HUC-8, HUC-10, and HUC-12 using identical
Iowa-clipped boundaries and the monitoring observations retained by
the final load-estimation pipeline. We required at least
{MIN_DAYS_PER_SITE_MONTH} valid daily discharge observations per
gauge-month and paired flow and nitrate only when both were available
in the same watershed and calendar month. For annual feasibility, all
12 months within a watershed-year were required to be paired.
Watersheds meeting this annual requirement represented
{h8['pct_area_with_complete_huc_year']:.1f}% of Iowa at HUC-8,
compared with {h10['pct_area_with_complete_huc_year']:.1f}% at HUC-10
and {h12['pct_area_with_complete_huc_year']:.1f}% at HUC-12. The
corresponding county reach was
{int(h8['counties_reached_complete_huc_year'])},
{int(h10['counties_reached_complete_huc_year'])}, and
{int(h12['counties_reached_complete_huc_year'])} counties. HUC-8 was
therefore retained as the primary statewide observational scale;
finer scales remain appropriate for local or model-assisted analyses
with denser paired monitoring.
""".strip()

print("\n" + "=" * 76)
print(methods_text)
print("\n" + "=" * 76)
print(results_text)
print("\n" + "=" * 76)
print(reviewer_text)

with open(
    OUT / "huc_scale_methods_FINAL.txt",
    "w",
    encoding="utf-8",
) as file:
    file.write(methods_text + "\n")

with open(
    OUT / "huc_scale_results_FINAL.txt",
    "w",
    encoding="utf-8",
) as file:
    file.write(results_text + "\n")

with open(
    OUT / "huc_scale_reviewer_response_FINAL.txt",
    "w",
    encoding="utf-8",
) as file:
    file.write(reviewer_text + "\n")


# ==============================================================
# 10. FINAL ACCEPTANCE CHECKS
# ==============================================================

assert FLOW_POINTS["site_no"].nunique() <= 166, (
    "Qualifying flow network unexpectedly exceeds "
    "the 166 contributing gauges."
)

assert len(GCONC) == 38_335, (
    "Nitrate observations do not match Pipeline v4."
)

complete_rows = SPECTRUM.loc[
    SPECTRUM["criterion"].eq(
        "12 paired months (complete)"
    )
]

assert len(complete_rows) == 3, (
    "Complete-year results were not produced for all scales."
)

assert (
    complete_rows["minimum_paired_months"] == 12
).all(), (
    "Complete HUC-years were not based on 12 paired months."
)

print("\n" + "=" * 76)
print("FINAL ACCEPTANCE CHECK: PASSED")
print("The HUC-scale analysis is now frozen.")
print("Route B remains the manuscript framing.")
print("=" * 76)


METHODS — HUC-SCALE JUSTIFICATION

We evaluated HUC-8, HUC-10, and HUC-12 as candidate observational
scales after clipping all watershed layers to the same Iowa boundary.
Streamflow support was restricted to gauges that contributed valid
daily discharge during 2000–2020, and a gauge-month was retained only
when at least 25 valid daily observations were
available. Nitrate observations were filtered using the same criteria
as the final county-load pipeline, retaining 38,335 Iowa DNR
and USGS observations. A paired watershed-month required qualifying
streamflow and nitrate observations in the same hydrologic unit,
calendar year, and month. A complete watershed-year required all
12 months to be paired. Annual feasibility was evaluated over
2001–2019.

RESULTS — HUC-SCALE JUSTIFICATION

Spatial co-location of qualifying streamflow and nitrate monitoring
covered 92.4% of Iowa at HUC-8,
40.5% at HUC-10, and
8.6% at HUC-12. Under the stricter annual
criterion requiring at least one complete 1